# TOTeM models

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LukasLiss/totem-tool/blob/main/totem_lib/docs/guide/totem.ipynb)

A TOTeM (Temporal Object Type Model) describes how the object types of a log relate (Liss et al., 2024). For each pair of connected object types, it has three relations in each direction: a temporal relation, a log cardinality and an event cardinality. This notebook discovers a TOTeM model from a tiny log, reads its relations, checks the model against the log, and saves it as JSON.

## Setup

In Google Colab, the first cell installs totem-lib from PyPI, where it is called `totem-tool`. For normal usage, install totem-lib first as shown in [Installation](../installation.md).

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install --quiet totem-tool "pulp<4"

The example log comes from the totem-lib repository. Outside a clone of the repository, the next cell downloads it (11 KB).

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

LOG = Path("../../test_data/small/resource_aware_orders.json")
if not LOG.exists():
    LOG = Path("resource_aware_orders.json")
    if not LOG.exists():
        urlretrieve(
            "https://raw.githubusercontent.com/LukasLiss/totem-tool/main/"
            "totem_lib/test_data/small/resource_aware_orders.json",
            LOG,
        )

## Load the log

The log has four orders with six items, packed into three packages. One worker, `w1`, picks, packs and ships everything.

Discovery works on an `ObjectCentricEventLog` from `import_ocel`. Conformance checking works on a DuckDB database from `import_ocel_db`. So the next cell loads the log twice. [Event logs](event_logs.ipynb) explains both forms.

In [ ]:
from totem_lib import import_ocel
from totem_lib.ocel import import_ocel_db

ocel = import_ocel(LOG)
db = import_ocel_db(LOG)

## Discover a model

`totemDiscovery` implements the discovery algorithm of the paper. It counts how often each relation holds between the objects of two types. For each pair of types, it keeps the most precise relation that holds for at least the share `tau` of the observations. With `tau=1.0`, a relation must always hold. A lower `tau` allows some noise in the log.

`totemDiscovery` prints progress messages while it runs. This page hides them.

In [ ]:
from totem_lib import totemDiscovery

totem = totemDiscovery(ocel, tau=0.9)

`totem_to_dict` turns the model into plain Python data.

In [ ]:
import polars as pl
from totem_lib import totem_to_dict

pl.Config.set_tbl_rows(20)  # show every row of these small tables
model = totem_to_dict(totem)

## Temporal relations

The temporal relations compare the lifespans of two connected objects, from their first to their last event. They group Allen's interval relations (Allen, 1983). For a pair of types `from` and `to`:

- `D` (during): the `from` object lives within the lifespan of the `to` object
- `I` (precedes): the `from` object starts before the `to` object and also ends before it
- `P` (parallel): any order is allowed, so there is no clear order

The model lists every relation in one direction only. `Di` and `Ii` are `D` and `I` seen from the other type.

In [ ]:
pl.DataFrame(
    [
        {"from": source, "to": target, "relation": relation}
        for relation, pairs in model["tempgraph"].items()
        if relation != "nodes"
        for source, target in pairs
    ]
)

Each item lives within its order, and the worker is active during the whole log. An item starts with `place order` and ends when it is packed, which is when its package starts. So items precede packages.

## Cardinalities

The cardinalities say how many objects of type `to` go with one object of type `from`:

- `log_cardinality`: over the whole log. Two objects are connected if they share an event or have an object-to-object relation.
- `event_cardinality`: within a single event that has a `from` object

The log cardinality is `1`, `0...1` (at most one), `1..*` (at least one) or `0...*` (any number). The event cardinality can also be `0`. The paper writes `0..1` and `0..*`.

In [ ]:
pl.DataFrame(model["cardinalities"])

Each item belongs to exactly one order, and each order has one or more items. But most events of an item do not include its order: only `place order` has both. So in one event, an item has at most one order (`0...1`).

## Draw the model

`plot()` draws the whole model in one picture. It needs the Graphviz program `dot`, see [Installation](../installation.md).

- Each box is an object type.
- Each arrow is the temporal relation from one type to another. The arrowhead shows which one: a filled box for `D`, a triangle for `I` and two bars for `P`.
- The label is the log cardinality, with the event cardinality in brackets. The picture uses the notation of the paper, so `0...1` becomes `0..1`.

In [ ]:
totem.plot()

## Check the model against a log

`conformance_of_totem` compares a model with a log. The paper does not cover this step; it is an addition of totem-lib. It checks the three relations separately and reports two numbers for each:

- **fitness**: the share of observations that the model relation allows. This is the score that discovery compares with `tau`.
- **precision**: how strict the model relation is. If a more precise relation would also allow most observations, precision is low.

Here, the model is checked against the log it was discovered from. In practice, you can check a model against a newer log of the same process.

In [ ]:
from totem_lib import conformance_of_totem

result = conformance_of_totem(totem, db)
pl.DataFrame(
    [{"relation": name, **metrics} for name, metrics in result.overall_metrics.to_dict().items()]
)

The model allows everything in the log, so all fitness values are 1. The precision of the event cardinalities is low. `type_pair_metrics` shows the result per pair of types, and `object_type_metrics` averages it per object type:

In [ ]:
pl.DataFrame(
    [
        {
            "from": pair.source_type,
            "to": pair.target_type,
            "event_cardinality": pair.event_cardinality.model_relation,
            "fitness": pair.event_cardinality.fitness,
            "precision": pair.event_cardinality.precision,
        }
        for pair in result.type_pair_metrics
    ]
)

The histograms in `result.histograms` show where these numbers come from. For items and orders:

In [ ]:
result.histograms.event_cardinality[("item", "order")]

Items take part in 13 events. Nine of them have no order, and four have one order. An event counts for every cardinality that allows it, so the counts overlap. The model relation `0...1` allows all 13 events, so the fitness is 1. But the more precise relation `0` already allows 9 of them, so the precision is only 1 - 9/13 = 0.31.

## Save and load a model

The dict from `totem_to_dict` contains only JSON data, so you can save it with the `json` module. `totem_from_dict` turns it back into a model.

In [ ]:
import json
import tempfile
from totem_lib import totem_from_dict

path = Path(tempfile.mkdtemp()) / "totem.json"
path.write_text(json.dumps(model, indent=2))

loaded = totem_from_dict(json.loads(path.read_text()))
totem_to_dict(loaded) == model

## Check your own model

A model does not have to come from discovery. You can write down the expected behavior, or change a discovered model, and check a log against it. For example, suppose that every order should have exactly one item:

In [ ]:
import copy

expected = copy.deepcopy(model)
for cardinality in expected["cardinalities"]:
    if (cardinality["from"], cardinality["to"]) == ("order", "item"):
        cardinality["log_cardinality"] = "1"

expected_result = conformance_of_totem(totem_from_dict(expected), db)
for pair in expected_result.type_pair_metrics:
    if (pair.source_type, pair.target_type) == ("order", "item"):
        print(pair.log_cardinality)

Only two of the four orders have one item, so the fitness is 0.5. Nothing is more precise than `1`, so the precision is 1. Over all pairs, the stricter model fits the log less well but is more precise:

In [ ]:
pl.DataFrame(
    [
        {"model": "discovered", **result.overall_metrics.log_cardinality.to_dict()},
        {"model": "expected", **expected_result.overall_metrics.log_cardinality.to_dict()},
    ]
)

`totem_from_dict` checks the data first, with `validate_totem_dict`. For an unknown relation, it raises a `ValueError`:

In [ ]:
expected["cardinalities"][0]["log_cardinality"] = "1..n"
try:
    totem_from_dict(expected)
except ValueError as error:
    print(error)

## Next steps

- [Getting started](../examples/getting_started.ipynb) and [Process areas](process_areas.md) use a TOTeM model to group the object types into process areas.
- The [API reference](../api/totem.rst) lists all functions for TOTeM models.

## References

- Liss, L., Adams, J.N., van der Aalst, W.M.P. (2024). TOTeM: Temporal Object Type Model for Object-Centric Process Mining. BPM Forum 2024, Springer.
- Allen, J.F. (1983). Maintaining Knowledge about Temporal Intervals. Communications of the ACM 26(11), 832–843.